# Lab 05 Prelab Walkthrough — Boolean Masks and Fitting a Saturating Amplifier

Work through this notebook **before** your Lab 05 session, running every cell
and reading every explanation. It teaches this lab's *new* Python skills on a
**simulated amplifier** — no hardware needed. The **CHECKPOINT** boxes ask
for specific values you will report in the **Prelab quiz on Canvas**.

Skills covered:

1. Simulating an amplifier that **saturates** against its supply rails
2. **Boolean masks** — selecting data by condition, and indexing arrays with them
3. What silently happens to a line fit when saturated points are left in
4. Writing a function that **returns several values** (tuple unpacking), and
   running the Lab 02 fit statistics on a masked subset

Run the setup cell first:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['font.size'] = 10

## 1. A simulated amplifier — with rails

In lab you will sweep a real amplifier's input and watch its output **clip**
near the supply rails. Here is a pretend amplifier with gain 4 whose output
cannot leave ±8.5 V, plus a little measurement noise (seeded, so everyone
gets identical numbers):

In [ ]:
rng = np.random.default_rng(42)

G_true   = 4.0                              # the amplifier's true gain
V_RAIL   = 8.5                              # output ceiling (V)

v_in  = np.linspace(-3.0, 3.0, 31)          # input sweep
v_out = np.clip(G_true * v_in, -V_RAIL, V_RAIL)   # gain, then the rails
v_out = v_out + rng.normal(0, 0.01, v_in.size)    # measurement noise

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(v_in, v_out, 'ro', markersize=5, label='simulated measurement')
ax.plot(v_in, G_true * v_in, 'b--', linewidth=1, label='ideal 4x amplifier')
ax.set_xlabel('$V_{in}$ (V)')
ax.set_ylabel('$V_{out}$ (V)')
ax.legend()
ax.grid(linestyle='--', linewidth=0.5)
plt.show()

`np.clip(x, lo, hi)` squashes everything outside `[lo, hi]` to the
boundary — a one-line supply rail. Note the shape: a straight line with the
ends folded flat. Your real LM358 data will look exactly like this.

> **CHECKPOINT 1:** Using gain 4 and rails at ±8.5 V, at what input voltage
> does this amplifier *start* to saturate? Report the value in volts
> (3 decimals).

## 2. Boolean masks — selecting data by condition

The flat ends are real data, but they are the wrong *regime* for a gain fit.
We want "only the points where the output is comfortably away from the
rails." NumPy expresses that as a **boolean mask**:

In [ ]:
linear = np.abs(v_out) < 7.9        # True/False, one per point

print(linear[:8], '...')             # the first few entries
print('type of each entry:', linear.dtype)
print('points that survive:', np.count_nonzero(linear), 'of', linear.size)

Comparing an array to a number compares **every element at once** and
returns an array of `True`/`False`. Feed that array back in as an index and
NumPy keeps only the `True` positions:

In [ ]:
v_in_lin  = v_in[linear]            # only the linear-region inputs
v_out_lin = v_out[linear]

print('kept inputs run from', v_in_lin.min(), 'to', v_in_lin.max())

This *mask indexing* idiom — build a condition, index with it — is one
of the most-used moves in all of scientific Python. In lab your condition
will be `np.abs(v_out) < 9.5` (real rails near ±10.5 V).

> **CHECKPOINT 2:** How many of the 31 sweep points survive the
> `np.abs(v_out) < 7.9` mask? Report the integer.

## 3. The fit that lies — and the fit that doesn't

Fit the line twice: once on everything, once on the masked subset:

In [ ]:
coeffs_all = np.polyfit(v_in, v_out, 1)              # WRONG: rails included
coeffs_lin = np.polyfit(v_in_lin, v_out_lin, 1)      # right: linear region

print(f"slope, all points   : {coeffs_all[0]:.4f}   (true gain is 4.0000)")
print(f"slope, masked points : {coeffs_lin[0]:.4f}")

The unmasked fit is *confidently wrong*: `polyfit` happily averages the
flat ends into the slope and reports a gain that describes nothing — not the
linear region, not the saturation, nothing. No error, no warning. The only
defense is looking at your data and masking deliberately. (Lab logbook
question Q12 has you repeat this stunt on your real data.)

> **CHECKPOINT 3:** Report both slopes from the cell above, each to 4
> decimals.

## 4. Functions that return several values

In lab you analyze two amplifiers identically — a job for a function
(`def`, Lab 02). New here: returning *several* results at once. Python packs
comma-separated return values into a tuple, and you unpack them with
matching names on the left of the `=`:

In [ ]:
def fit_line_stats(x, y, mask):
    """Fit y vs x on the masked points; return slope, CI, and s_yx."""
    coeffs = np.polyfit(x[mask], y[mask], 1)

    N      = np.count_nonzero(mask)
    nu     = N - 2
    resid  = y[mask] - np.polyval(coeffs, x[mask])
    s_yx   = np.sqrt(np.sum(resid**2)) / np.sqrt(nu)
    S_m    = s_yx / np.sqrt(np.sum((x[mask] - x[mask].mean())**2))
    CI_m   = stats.t.ppf(0.975, df=nu) * S_m     # 95% CI on the slope

    return coeffs[0], CI_m, s_yx                 # three values, one return


slope, CI, s_yx = fit_line_stats(v_in, v_out, linear)   # three names unpack it
print(f"G = {slope:.4f} ± {CI:.4f} (95% CI),  s_yx = {s_yx:.4f} V")

Everything inside the function is Lab 02's fit-statistics recipe — the
only news is `x[mask]` doing the selecting and the three-value `return`.
The lab's `fit_gain` function is this, grown two sizes.

> **CHECKPOINT 4:** Report the 95% CI half-width on the slope (the `± ...`
> number) to 4 decimals.

## Summary: where each skill is used in lab

| Skill | Where you will use it |
|---|---|
| recognizing saturation in a transfer curve | Parts 2–5, post-lab Q2/Q3 |
| boolean mask + `array[mask]` indexing | selecting the linear region in `fit_gain` |
| `np.count_nonzero(mask)` | the correct N for your fit statistics |
| masked `polyfit` + CI (Lab 02 recipe) | measuring both amplifier gains |
| multi-value `return` + unpacking | the `fit_gain` function you'll write |

See you in lab — bring your golden rules.